<a href="https://colab.research.google.com/github/simplyshree/SeqTrainer/blob/issue-3-all-model-baselines/notebooks/final_training/dnabert2_final_training_t4_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DNABERT2 final staged training on a Colab T4

This notebook is the final T4-compatible DNABERT2 candidate for the shared
SeqTrainer promoter benchmark. It stages all inputs on local Colab storage,
uses validation MCC for every decision, saves resumable state to Drive, and
evaluates the held-out test split only after final candidate selection.

## Fixed contract

- GSE144621 EP_DNA_BERT2_genomic_order
- exact shared train, validation, and test CSV files
- labels 0 = non-promoter and 1 = promoter
- seed 42
- DNABERT-2-117M revision 7bce263b15377fc15361f52cfab88f8b586abda0
- model selection and threshold selection on validation MCC only
- validation AUPRC is the tie-breaker
- test is evaluated once at the end
- output artifacts are written under SeqTrainer/final_training/dnabert2_seed42

In [ ]:
# 1. Verify the requested accelerator.
import subprocess

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True,
).strip()
print(gpu_info)
if "T4" not in gpu_info:
    raise RuntimeError("Select an NVIDIA T4 runtime, then rerun this notebook from the top.")

In [ ]:
# 2. Runtime configuration. Change only DRIVE_DATA_DIR if your Drive layout differs.
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-all-model-baselines"
REPO_DIR = Path("/content/SeqTrainer")
MINIFORGE_DIR = Path("/content/miniforge3")
ENV_DIR = Path("/content/envs/seqtrainer-final-dnabert2-t4")
ENV_PYTHON = ENV_DIR / "bin" / "python"

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AIxBio/Promoter Classification/Data")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/SeqTrainer/final_training/dnabert2_seed42")
DRIVE_MODEL_CACHE = Path("/content/drive/MyDrive/SeqTrainer/final_training/model_cache/dnabert2")
LOCAL_DATA_DIR = Path("/content/seqtrainer_final_training/data")
LOCAL_OUTPUT_DIR = Path("/content/seqtrainer_final_training/dnabert2_seed42")
LOCAL_HF_HOME = Path("/content/seqtrainer_final_training/huggingface")

RESUME_MODE = "latest"  # latest, best, or none
LOSS_MODE = "bce"       # bce or focal
RUN_MAX_LENGTH_128_CANDIDATE = False
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive data:", DRIVE_DATA_DIR)
print("Drive output:", DRIVE_OUTPUT_DIR)

In [ ]:
# 3. Create the pinned Python 3.10 environment.
installer = Path("/content/Miniforge3-Linux-x86_64.sh")
if not (MINIFORGE_DIR / "bin" / "conda").exists():
    subprocess.run(
        ["wget", "-q", "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh", "-O", str(installer)],
        check=True,
    )
    subprocess.run(["bash", str(installer), "-b", "-p", str(MINIFORGE_DIR)], check=True)
conda = MINIFORGE_DIR / "bin" / "conda"
if not ENV_PYTHON.exists():
    subprocess.run([str(conda), "create", "-y", "-p", str(ENV_DIR), "python=3.10", "pip"], check=True)
print("Environment:", ENV_PYTHON)

In [ ]:
# 4. Check out the requested branch without touching main or annotation-mvp.
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
print("Branch:", subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# 5. Install the pinned T4 environment.
torch_index = "https://download.pytorch.org/whl/cu121"
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools<70", "wheel"], check=True)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "torch==2.2.2", "--index-url", torch_index], check=True)
subprocess.run([
    str(ENV_PYTHON), "-m", "pip", "install",
    "transformers==4.29.2", "numpy==1.24.4", "pandas==2.0.3",
    "scikit-learn==1.3.2", "einops==0.6.1", "rdflib", "matplotlib",
], check=True)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)
print("Pinned packages installed.")

In [ ]:
# 6. Mount Drive safely after a reconnect and restore the model cache.
from google.colab import drive
import os
import time

MOUNT_POINT = Path("/content/drive")
already_mounted = os.path.ismount(str(MOUNT_POINT)) or (MOUNT_POINT / "MyDrive").exists()

if already_mounted:
    print("Google Drive is already mounted.")
else:
    if MOUNT_POINT.exists() and not MOUNT_POINT.is_symlink() and any(MOUNT_POINT.iterdir()):
        stale_dir = Path("/content/drive_stale")
        if stale_dir.exists():
            stale_dir = Path(f"/content/drive_stale_{int(time.time())}")
        MOUNT_POINT.rename(stale_dir)
        print("Moved stale mount directory to:", stale_dir)
    MOUNT_POINT.mkdir(parents=True, exist_ok=True)
    drive.mount(str(MOUNT_POINT), force_remount=False)
    print("Google Drive mounted.")

DRIVE_MODEL_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
if any(DRIVE_MODEL_CACHE.iterdir()):
    shutil.copytree(DRIVE_MODEL_CACHE, LOCAL_HF_HOME, dirs_exist_ok=True)
    print("Restored DNABERT2 cache from Drive.")
else:
    print("No Drive model cache yet; the first run will download the pinned revision.")
print("Local HF cache:", LOCAL_HF_HOME)

In [ ]:
# 7. Verify the isolated package and GPU before the long run.
check = r'''
import json, sys, torch, transformers, seqtrainer
print(json.dumps({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seqtrainer": seqtrainer.__file__,
}, indent=2))
assert torch.cuda.is_available()
assert "T4" in torch.cuda.get_device_name(0)
'''
subprocess.run([str(ENV_PYTHON), "-c", check], check=True)

## 8. Run staged training

The runner audits the exact Drive directory, atomically stages local CSVs,
creates the token-length audit, trains with FP16 and gradient accumulation,
appends history after every epoch, and saves latest/best checkpoints
atomically to Drive. Rerunning this cell with RESUME_MODE = "latest" continues
after a disconnect.

In [ ]:
import json

helper = REPO_DIR / "notebooks/final_training/helpers/run_dnabert2_final.py"
run_env = os.environ.copy()
run_env["HF_HOME"] = str(LOCAL_HF_HOME)
run_env["TRANSFORMERS_CACHE"] = str(LOCAL_HF_HOME)
run_env["TOKENIZERS_PARALLELISM"] = "false"
run_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def run_candidate(max_length, drive_output, local_output, selection_only):
    command = [
        str(ENV_PYTHON), str(helper),
        "--repo-dir", str(REPO_DIR),
        "--drive-data-dir", str(DRIVE_DATA_DIR),
        "--local-data-dir", str(LOCAL_DATA_DIR),
        "--drive-output-dir", str(drive_output),
        "--local-output-dir", str(local_output),
        "--resume", RESUME_MODE,
        "--loss-mode", LOSS_MODE,
        "--max-length", str(max_length),
    ]
    if selection_only:
        command.append("--selection-only")
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, env=run_env)

if RUN_MAX_LENGTH_128_CANDIDATE:
    # Both candidates are validation-only. Test is run only for the winner.
    candidate_root_drive = DRIVE_OUTPUT_DIR / "candidates"
    candidate_root_local = LOCAL_OUTPUT_DIR / "candidates"
    for max_length in (104, 128):
        run_candidate(
            max_length,
            candidate_root_drive / f"max_length_{max_length}",
            candidate_root_local / f"max_length_{max_length}",
            selection_only=True,
        )
    candidate_scores = []
    for max_length in (104, 128):
        candidate_metrics = json.loads(
            (candidate_root_drive / f"max_length_{max_length}" / "metrics.json").read_text()
        )
        validation = candidate_metrics["validation"]
        candidate_scores.append((float(validation["mcc"]), float(validation["auprc"]), max_length))
    selected_length = max(candidate_scores)[2]
    print("Validation-only max-length selection:", candidate_scores, "selected:", selected_length)
    run_candidate(selected_length, DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR, selection_only=False)
else:
    run_candidate(104, DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR, selection_only=False)

# Keep the downloaded revision reusable without making Drive the active loader.
if LOCAL_HF_HOME.exists():
    shutil.copytree(LOCAL_HF_HOME, DRIVE_MODEL_CACHE, dirs_exist_ok=True)
    print("DNABERT2 cache synced to:", DRIVE_MODEL_CACHE)

## 9. Create final diagnostic plots

This cell runs only after the validation-selected checkpoint is locked. It
does not choose a model or threshold using test labels.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve

metrics = pd.read_csv(DRIVE_OUTPUT_DIR / "metrics.csv")
predictions = pd.read_csv(DRIVE_OUTPUT_DIR / "predictions.csv")
plot_dir = DRIVE_OUTPUT_DIR / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, split in zip(axes, ["train", "validation", "test"]):
    values = predictions[predictions["split"] == split]
    ConfusionMatrixDisplay.from_predictions(
        values["label"], values["prediction"], labels=[0, 1],
        display_labels=["non-promoter", "promoter"], colorbar=False, ax=axis,
    )
    axis.set_title(split)
fig.tight_layout()
fig.savefig(plot_dir / "confusion_matrices.png", dpi=160)
plt.close(fig)

test_values = predictions[predictions["split"] == "test"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fpr, tpr, _ = roc_curve(test_values["label"], test_values["probability"])
precision, recall, _ = precision_recall_curve(test_values["label"], test_values["probability"])
axes[0].plot(fpr, tpr)
axes[0].set(title="Held-out test ROC", xlabel="false positive rate", ylabel="true positive rate")
axes[1].plot(recall, precision)
axes[1].set(title="Held-out test precision-recall", xlabel="recall", ylabel="precision")
fig.tight_layout()
fig.savefig(plot_dir / "test_roc_pr_curves.png", dpi=160)
plt.close(fig)

validation = predictions[predictions["split"] == "validation"]
thresholds = np.unique(np.r_[0.0, validation["probability"].to_numpy(), 1.0])
from sklearn.metrics import matthews_corrcoef
threshold_mcc = [
    matthews_corrcoef(validation["label"], (validation["probability"] >= threshold).astype(int))
    for threshold in thresholds
]
plt.figure(figsize=(9, 4))
plt.plot(thresholds, threshold_mcc)
plt.xlabel("validation threshold")
plt.ylabel("validation MCC")
plt.title("Exact validation threshold search")
plt.tight_layout()
plt.savefig(plot_dir / "validation_threshold_mcc.png", dpi=160)
plt.close()
print("Plots saved in:", plot_dir)

In [ ]:
# 10. Verify all required final artifacts.
required = [
    "config.json", "input_split_audit.json", "environment.json", "history.csv",
    "metrics.csv", "metrics.json", "predictions.csv", "manifest.json",
    "checkpoints/latest.pt", "checkpoints/best_validation_mcc.pt",
    "plots/confusion_matrices.png", "plots/test_roc_pr_curves.png",
    "plots/validation_threshold_mcc.png",
]
missing = [name for name in required if not (DRIVE_OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError("Missing final DNABERT2 artifacts: " + ", ".join(missing))
display(metrics)
print("Held-out test MCC:", float(metrics.loc[metrics.split == "test", "mcc"].iloc[0]))
print("Held-out test AUPRC:", float(metrics.loc[metrics.split == "test", "auprc"].iloc[0]))
print("Outputs:", DRIVE_OUTPUT_DIR)

## Final step: verify outputs and disconnect runtime

Do not place executable cells after this cell.

In [ ]:
import os
import time

required_outputs = [
    DRIVE_OUTPUT_DIR / "metrics.csv",
    DRIVE_OUTPUT_DIR / "metrics.json",
    DRIVE_OUTPUT_DIR / "predictions.csv",
    DRIVE_OUTPUT_DIR / "manifest.json",
    DRIVE_OUTPUT_DIR / "history.csv",
]
missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    raise FileNotFoundError("Runtime will not disconnect because outputs are missing: " + ", ".join(missing))
print("All required artifacts were saved:")
for path in required_outputs:
    print("-", path)
if hasattr(os, "sync"):
    os.sync()
print("Disconnecting the Colab runtime in five seconds.")
time.sleep(5)
from google.colab import runtime
runtime.unassign()